[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bsheese/225/blob/main/10_pandas_sql/10_1_Query.ipynb)

# 10.1: `df.query()`: SQL-Like Filtering in pandas

You already know how to filter a DataFrame with boolean indexing: `df[df["lifeExp"] > 70]`. For a single condition that works well. But as soon as you need two or three conditions at once, the syntax starts to fight you. This notebook introduces `df.query()`, a cleaner way to express filter conditions that also happens to look a lot like the database query language you will learn in the next few notebooks.

In [1]:
import pandas as pd

url = "https://raw.githubusercontent.com/jennybc/gapminder/main/inst/extdata/gapminder.tsv"
df = pd.read_csv(url, sep="\t")
df.head()

,country,continent,year,lifeExp,pop,gdpPercap
0,Afghanistan,Asia,1952,28.801,8425333,779.445314
1,Afghanistan,Asia,1957,30.332,9240934,820.853030
2,Afghanistan,Asia,1962,31.997,10267083,853.100710
3,Afghanistan,Asia,1967,34.020,11537966,836.197138
4,Afghanistan,Asia,1972,36.088,13079460,739.981106


## The problem with chained boolean conditions

Suppose you want to find Asian countries in 2007 with a life expectancy above 70. That is three conditions. With boolean indexing, each condition needs its own pair of parentheses, and they chain together with `&` operators:

In [2]:
result_boolean = df[
    (df["year"] == 2007) &
    (df["continent"] == "Asia") &
    (df["lifeExp"] > 70)
]
result_boolean[["country", "lifeExp", "gdpPercap"]]

,country,lifeExp,gdpPercap
95,Bahrain,75.635,29796.048340
299,China,72.961,4959.114854
671,"Hong Kong, China",82.208,39724.978670
719,Indonesia,70.650,3540.651564
731,Iran,70.964,11605.714490
767,Israel,80.745,25523.277100
803,Japan,82.603,31656.068060
815,Jordan,72.535,4519.461171
851,"Korea, Rep.",78.623,23348.139730
863,Kuwait,77.588,47306.989780


The result is correct: 22 Asian countries had a life expectancy above 70 in 2007. But the code is hard to read at a glance. The actual logic, year equals 2007, continent is Asia, life expectancy above 70, is buried inside bracket nesting that has nothing to do with the filtering logic itself. With four or five conditions this becomes genuinely difficult to scan.

There is nothing wrong with this syntax. It works, and for a single condition it is the natural choice. But for complex filters, `df.query()` offers the same result in a form that reads more like the question you are actually asking.

## `df.query()`: filtering with a string expression

`query()` takes a single string describing the filter condition. The string uses column names directly, with no `df["col"]` wrapper and no outer parentheses around each condition. The keyword `and` replaces `&`, and `or` replaces `|`.

In [3]:
result_query = df.query("year == 2007 and continent == 'Asia' and lifeExp > 70")
result_query[["country", "lifeExp", "gdpPercap"]]

,country,lifeExp,gdpPercap
95,Bahrain,75.635,29796.048340
299,China,72.961,4959.114854
671,"Hong Kong, China",82.208,39724.978670
719,Indonesia,70.650,3540.651564
731,Iran,70.964,11605.714490
767,Israel,80.745,25523.277100
803,Japan,82.603,31656.068060
815,Jordan,72.535,4519.461171
851,"Korea, Rep.",78.623,23348.139730
863,Kuwait,77.588,47306.989780


The output is identical to the boolean-indexing version: same 22 rows, same values. A few things to notice about the syntax:

- String values inside the filter expression use single quotes (`'Asia'`), because the expression itself is already inside double quotes.
- The comparison operators are the same as Python: `==`, `!=`, `<`, `>`, `<=`, `>=`.
- `and` and `or` are the plain English words, not `&` and `|`.

For a three-condition filter the readability improvement is already visible. The logic reads almost like a sentence: year is 2007, continent is Asia, life expectancy above 70.

## Injecting Python variables with `@`

Sometimes the threshold in a filter is not a hard-coded number but the result of a computation: the median, the mean, or a value the user supplies. Because the filter condition lives inside a string, you cannot just type a variable name; `query()` would treat it as a column name and fail.

The fix is the `@` prefix: `@varname` tells `query()` to look up that name in the surrounding Python environment rather than treating it as a column.

In [4]:
# Find countries above the global median life expectancy in 2007
median_life = df.query("year == 2007")["lifeExp"].median()
print(f"Global median life expectancy in 2007: {median_life:.1f} years")

above_median = df.query("year == 2007 and lifeExp > @median_life")
print(f"Countries above median: {len(above_median)}")
above_median[["country", "continent", "lifeExp"]].head(10)

Global median life expectancy in 2007: 71.9 years
Countries above median: 71


,country,continent,lifeExp
23,Albania,Europe,76.423
35,Algeria,Africa,72.301
59,Argentina,Americas,75.320
71,Australia,Oceania,81.235
83,Austria,Europe,79.829
95,Bahrain,Asia,75.635
119,Belgium,Europe,79.441
155,Bosnia and Herzegovina,Europe,74.852
179,Brazil,Americas,72.390
191,Bulgaria,Europe,73.005


The median life expectancy across all 142 countries in 2007 was about 71.9 years. Exactly half the countries fall above it. The `@median_life` in the query string is replaced at run time with the computed value. This means you can compute a threshold in one line and reference it in the query in the next line without copying the number by hand.

## Filtering on string columns

String comparisons work the same way as numeric ones. The value you are comparing against goes in single quotes inside the string expression. You can also use `in` to match against a list of values, which is cleaner than chaining multiple `or` conditions.

In [5]:
# Single string condition
africa_2007 = df.query("continent == 'Africa' and year == 2007")
print(f"African countries in 2007: {len(africa_2007)}")

# 'in' to match multiple values
nordic = df.query("country in ['Norway', 'Sweden', 'Denmark', 'Finland', 'Iceland']")
print(f"Nordic country-years: {len(nordic)}")
nordic.groupby("country")["lifeExp"].agg(["min", "max"]).round(1)

African countries in 2007: 52
Nordic country-years: 60


,min,max
country,,
Denmark,70.8,78.3
Finland,66.6,79.3
Iceland,72.5,81.8
Norway,72.7,80.2
Sweden,71.9,80.9


There are 52 African countries in the 2007 snapshot. The Nordic subset pulls all 12 years for each of the five countries. Using `in` with a list is much more readable than five separate `or continent == ...` conditions chained together, and it is directly parallel to what you would write in a database query, as you will see in notebook 10.3.

## Negation with `not`

To exclude a value or group, use `not` inside the query string. It works with both single conditions and `in` lists.

In [6]:
# All continents except Oceania
no_oceania = df.query("year == 2007 and continent != 'Oceania'")
print("Continents:", sorted(no_oceania["continent"].unique()))

# Equivalent using 'not in'
no_oceania2 = df.query("year == 2007 and continent not in ['Oceania']")
print("Same result:", no_oceania.shape == no_oceania2.shape)

Continents: ['Africa', 'Americas', 'Asia', 'Europe']
Same result: True


Both forms give the same 140-country result. For a single exclusion `!=` is shorter. For excluding several values at once, `not in [...]` is cleaner than chaining multiple `!=` conditions.

## Chaining `query()` calls

`query()` returns a regular DataFrame, so you can chain a second `query()` call directly onto the result, or chain any other pandas method. This is sometimes useful when you want to build up a filter in stages and keep each stage readable.

In [7]:
# Two-stage filter: first narrow to 2007, then apply the substantive condition
result_chained = (
    df
    .query("year == 2007")
    .query("lifeExp > 75 and gdpPercap < 20000")
    [["country", "continent", "lifeExp", "gdpPercap"]]
    .sort_values("lifeExp", ascending=False)
)
result_chained

,country,continent,lifeExp,gdpPercap
359,Costa Rica,Americas,78.782,9645.061420
1259,Puerto Rico,Americas,78.746,19328.709010
287,Chile,Americas,78.553,13171.638850
395,Cuba,Americas,78.273,8948.102923
1271,Reunion,Africa,76.442,7670.122558
23,Albania,Europe,76.423,5937.029526
1631,Uruguay,Americas,76.384,10611.462990
995,Mexico,Americas,76.195,11977.574960
383,Croatia,Europe,75.748,14619.222720
1235,Poland,Europe,75.563,15389.924680


These are countries that had long life expectancy in 2007 without very high GDP, suggesting strong public health outcomes independent of raw wealth. The two-stage chain is equivalent to combining both conditions in a single query, but splitting them can help when the first filter is a standard setup step (isolating a year) and the second is the question you are actually investigating.

## When to use `query()` vs boolean indexing

Both tools filter rows and both are correct. The practical difference is readability:

- For one or two conditions, boolean indexing is fine: `df[df["year"] == 2007]` is clear and concise.
- For three or more conditions, `query()` is usually easier to read because the logic sits in plain text without bracket nesting.
- When you need the result of a computation as the threshold, `@variable` injection makes `query()` more convenient than storing the value and re-referencing it in a boolean expression.
- Boolean indexing is sometimes necessary when the condition involves a pandas method that `query()` cannot express, like `str.contains()` with a regex pattern.

The two are not in competition. Use whichever is clearer for the filter you are writing.

## What's next

You have seen that `query()` expresses filter conditions as a readable string: column names, comparison operators, `and`/`or`/`not`, and `in` lists. That syntax is not a pandas invention. It is a deliberate approximation of SQL, the query language that almost every database in the world understands. In notebook 10.2, you will load the Gapminder data into a real database and write your first SQL queries, and you will notice immediately that the syntax you just learned carries over almost unchanged.